# 💬 Module 8: RAG AI Assistant (FAISS Vector Store & Sentence Transformers)

In this notebook, we construct a product and analytics knowledge base from retail metrics, encode text into vector embeddings using **Sentence Transformers (`all-MiniLM-L6-v2`)**, build a **FAISS Vector Database Index**, and query natural language business answers using context retrieval and **Flan-T5**.

---

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

cleaned_path = os.path.join("..", "data", "processed", "cleaned_retail_data.csv")
df = pd.read_csv(cleaned_path)
print(f"Loaded Cleaned Data: {df.shape[0]:,} rows")

C:\Users\jiten\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded Cleaned Data: 392,692 rows


### Step 1: Construct Knowledge Base Documents

In [2]:
total_rev = df['TotalPrice'].sum()
total_orders = df['InvoiceNo'].nunique()
total_cust = df['CustomerID'].nunique()
total_prod = df['StockCode'].nunique()

top_prods = df.groupby('Description')['TotalPrice'].sum().sort_values(ascending=False).head(10)
top_prod_str = ", ".join([f"{p} (£{r:,.2f})" for p, r in top_prods.items()])

docs = [
    {"topic": "summary", "content": f"The Online Retail Platform achieved total revenue of £{total_rev:,.2f} across {total_orders:,} orders, {total_cust:,} customers, and {total_prod:,} products."},
    {"topic": "top_products", "content": f"The top revenue products are: {top_prod_str}. Single highest item: '{top_prods.index[0]}' (£{top_prods.iloc[0]:,.2f})."}
]

for doc in docs:
    print(f"Topic: {doc['topic']}")
    print(f"Content: {doc['content']}\n")

Topic: summary
Content: The Online Retail Platform achieved total revenue of £8,887,208.89 across 18,532 orders, 4,338 customers, and 3,665 products.

Topic: top_products
Content: The top revenue products are: PAPER CRAFT , LITTLE BIRDIE (£168,469.60), REGENCY CAKESTAND 3 TIER (£142,264.75), WHITE HANGING HEART T-LIGHT HOLDER (£100,392.10), JUMBO BAG RED RETROSPOT (£85,040.54), MEDIUM CERAMIC TOP STORAGE JAR (£81,416.73), POSTAGE (£77,803.96), PARTY BUNTING (£68,785.23), ASSORTED COLOUR BIRD ORNAMENT (£56,413.03), Manual (£53,419.93), RABBIT NIGHT LIGHT (£51,251.24). Single highest item: 'PAPER CRAFT , LITTLE BIRDIE' (£168,469.60).



### Step 2: Generate Sentence Embeddings (`all-MiniLM-L6-v2`)

In [3]:
st_model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [doc['content'] for doc in docs]
embeddings = st_model.encode(texts, convert_to_numpy=True)
print(f"Generated Embeddings Matrix Shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1553.60it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated Embeddings Matrix Shape: (2, 384)


### Step 3: Build & Save FAISS Vector Database

In [4]:
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings.astype('float32'))

v_dir = os.path.join("..", "vector_db")
os.makedirs(v_dir, exist_ok=True)
faiss.write_index(index, os.path.join(v_dir, "faiss_index.bin"))
with open(os.path.join(v_dir, "product_metadata.pkl"), 'wb') as f:
    pickle.dump(docs, f)

print("FAISS vector store built and saved successfully!")

FAISS vector store built and saved successfully!


### Step 4: Semantic Query Test

In [5]:
query_text = "Which products generated highest revenue?"
q_emb = st_model.encode([query_text], convert_to_numpy=True)
distances, indices = index.search(q_emb.astype('float32'), k=1)

retrieved_content = docs[indices[0][0]]['content']
print(f"❓ Question: '{query_text}'")
print(f"💡 Retrieved Context:\n{retrieved_content}")

❓ Question: 'Which products generated highest revenue?'
💡 Retrieved Context:
The top revenue products are: PAPER CRAFT , LITTLE BIRDIE (£168,469.60), REGENCY CAKESTAND 3 TIER (£142,264.75), WHITE HANGING HEART T-LIGHT HOLDER (£100,392.10), JUMBO BAG RED RETROSPOT (£85,040.54), MEDIUM CERAMIC TOP STORAGE JAR (£81,416.73), POSTAGE (£77,803.96), PARTY BUNTING (£68,785.23), ASSORTED COLOUR BIRD ORNAMENT (£56,413.03), Manual (£53,419.93), RABBIT NIGHT LIGHT (£51,251.24). Single highest item: 'PAPER CRAFT , LITTLE BIRDIE' (£168,469.60).
